# Dashboard Interativo - Previsão de Consumo Energético

Dashboard com visualizações interativas das previsões e métricas dos modelos.

**Pré-requisito**: Execute primeiro o notebook `02_pipeline_modeling.ipynb` ou o script `src/pipeline.py` para gerar os dados processados e modelos.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas.')

Bibliotecas carregadas.


In [2]:
# Carregar dados
predictions = pd.read_parquet('../data/predictions.parquet')
features = pd.read_parquet('../data/features_processed.parquet')

# Carregar metadados do modelo
meta_files = [f for f in os.listdir('../models') if f.startswith('metadata_') and f.endswith('.json')]
if 'metadata_latest.json' in meta_files:
    meta_path = '../models/metadata_latest.json'
else:
    meta_path = '../models/' + sorted(meta_files)[-1]

with open(meta_path) as f:
    metadata = json.load(f)

print(f'Previsões: {predictions.shape}')
print(f'Features: {features.shape}')
print(f'Melhor modelo: {metadata["best_model"]}')
print(f'Versão: {metadata["version"]}')

Previsões: (2900, 25)
Features: (17300, 23)
Melhor modelo: lightgbm
Versão: 20260210_110144


## 1. Métricas dos Modelos

In [3]:
# Comparação de métricas entre modelos
metrics = metadata['metrics']
metric_names = ['MAE (kWh)', 'RMSE (kWh)', 'R²', 'MAPE (%)']
metric_keys = ['mae', 'rmse', 'r2', 'mape']

fig = make_subplots(rows=1, cols=4, subplot_titles=metric_names)

for i, (name, key) in enumerate(zip(metric_names, metric_keys)):
    rf_val = metrics['random_forest'][key]
    lgb_val = metrics['lightgbm'][key]
    
    fig.add_trace(
        go.Bar(name='Random Forest' if i == 0 else '', x=['RF'], y=[rf_val],
               marker_color='steelblue', showlegend=(i == 0), legendgroup='RF'),
        row=1, col=i+1
    )
    fig.add_trace(
        go.Bar(name='LightGBM' if i == 0 else '', x=['LGB'], y=[lgb_val],
               marker_color='coral', showlegend=(i == 0), legendgroup='LGB'),
        row=1, col=i+1
    )

fig.update_layout(
    title_text='Comparação de Métricas - Random Forest vs LightGBM',
    height=400, barmode='group'
)
fig.show()

## 2. Previsões vs Valores Reais

In [4]:
# Série temporal: Previsão vs Real (agregado diário)
daily = predictions.groupby('date').agg(
    real=('consumption_kwh', 'mean'),
    rf_pred=('rf_prediction', 'mean'),
    lgb_pred=('lgb_prediction', 'mean')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.date, y=daily.real, name='Real',
                        line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=daily.date, y=daily.rf_pred, name='Random Forest',
                        line=dict(color='steelblue', dash='dash')))
fig.add_trace(go.Scatter(x=daily.date, y=daily.lgb_pred, name='LightGBM',
                        line=dict(color='coral', dash='dash')))

fig.update_layout(
    title='Consumo Médio Diário - Real vs Previsões (Período de Teste)',
    xaxis_title='Data', yaxis_title='Consumo Médio (kWh)',
    height=500, template='plotly_white'
)
fig.show()

In [5]:
# Scatter: Previsto vs Real
best = metadata['best_model']
pred_col = 'lgb_prediction' if best == 'lightgbm' else 'rf_prediction'

fig = px.scatter(
    predictions, x='consumption_kwh', y=pred_col,
    color='region', opacity=0.3,
    labels={'consumption_kwh': 'Real (kWh)', pred_col: 'Previsto (kWh)'},
    title=f'Previsto vs Real - {best.replace("_", " ").title()} por Região'
)
fig.add_trace(go.Scatter(
    x=[predictions.consumption_kwh.min(), predictions.consumption_kwh.max()],
    y=[predictions.consumption_kwh.min(), predictions.consumption_kwh.max()],
    mode='lines', line=dict(color='red', dash='dash'), name='Linha Ideal'
))
fig.update_layout(height=500, template='plotly_white')
fig.show()

## 3. Análise de Erros

In [6]:
# Distribuição dos erros
predictions['error_rf'] = predictions['consumption_kwh'] - predictions['rf_prediction']
predictions['error_lgb'] = predictions['consumption_kwh'] - predictions['lgb_prediction']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Random Forest', 'LightGBM'])

fig.add_trace(
    go.Histogram(x=predictions['error_rf'], nbinsx=50, marker_color='steelblue',
                 name='RF', opacity=0.8),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=predictions['error_lgb'], nbinsx=50, marker_color='coral',
                 name='LGB', opacity=0.8),
    row=1, col=2
)

fig.update_layout(title='Distribuição dos Erros de Previsão', height=400, template='plotly_white')
fig.update_xaxes(title_text='Erro (kWh)')
fig.show()

In [7]:
# Erro por região
error_by_region = predictions.groupby('region').agg(
    mae_rf=('error_rf', lambda x: np.abs(x).mean()),
    mae_lgb=('error_lgb', lambda x: np.abs(x).mean()),
    count=('client_id', 'count')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(name='Random Forest', x=error_by_region.region, y=error_by_region.mae_rf,
                     marker_color='steelblue'))
fig.add_trace(go.Bar(name='LightGBM', x=error_by_region.region, y=error_by_region.mae_lgb,
                     marker_color='coral'))

fig.update_layout(
    title='MAE por Região', xaxis_title='Região', yaxis_title='MAE (kWh)',
    barmode='group', height=400, template='plotly_white'
)
fig.show()

## 4. Análise Regional do Consumo

In [8]:
# Heatmap de consumo médio por região e semana
features['week'] = features['date'].dt.isocalendar().week.astype(int)
heatmap_data = features.groupby(['region', 'week'])['consumption_kwh'].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index='region', columns='week', values='consumption_kwh')

fig = px.imshow(
    heatmap_pivot, aspect='auto',
    labels=dict(x='Semana do Ano', y='Região', color='Consumo (kWh)'),
    title='Mapa de Calor - Consumo Médio por Região e Semana',
    color_continuous_scale='YlOrRd'
)
fig.update_layout(height=400, template='plotly_white')
fig.show()

In [9]:
# Box plot interativo por região
fig = px.box(
    features, x='region', y='consumption_kwh', color='region',
    title='Distribuição do Consumo por Região',
    labels={'consumption_kwh': 'Consumo (kWh)', 'region': 'Região'}
)
fig.update_layout(height=450, template='plotly_white', showlegend=False)
fig.show()

## 5. Consumo por Cliente (Explorador Interativo)

In [10]:
# Top 10 clientes por consumo médio
top_clients = features.groupby('client_id')['consumption_kwh'].mean().nlargest(10).index.tolist()

top_data = features[features.client_id.isin(top_clients)]

fig = px.line(
    top_data, x='date', y='consumption_kwh', color='client_id',
    title='Top 10 Clientes - Série Temporal de Consumo',
    labels={'consumption_kwh': 'Consumo (kWh)', 'date': 'Data', 'client_id': 'Cliente'}
)
fig.update_layout(height=500, template='plotly_white')
fig.show()

## 6. Impacto Climático no Consumo

In [11]:
# Scatter interativo: Temperatura vs Consumo vs Região
daily_features = features.groupby(['date', 'region']).agg(
    consumption_kwh=('consumption_kwh', 'mean'),
    temperature=('temperature', 'mean'),
    humidity=('humidity', 'mean')
).reset_index()

fig = px.scatter(
    daily_features, x='temperature', y='consumption_kwh',
    color='region', size='humidity', opacity=0.6,
    title='Consumo vs Temperatura (tamanho = umidade)',
    labels={'temperature': 'Temperatura (°C)', 'consumption_kwh': 'Consumo Médio (kWh)',
            'humidity': 'Umidade (%)', 'region': 'Região'},
    trendline='ols'
)
fig.update_layout(height=500, template='plotly_white')
fig.show()

In [12]:
# Evolução de temperatura e consumo ao longo do tempo
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Consumo Médio Diário', 'Temperatura Média Diária'])

daily_all = features.groupby('date').agg(
    consumption_kwh=('consumption_kwh', 'mean'),
    temperature=('temperature', 'mean')
).reset_index()

fig.add_trace(
    go.Scatter(x=daily_all.date, y=daily_all.consumption_kwh.rolling(7).mean(),
              name='Consumo (MM7d)', line=dict(color='steelblue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=daily_all.date, y=daily_all.temperature.rolling(7).mean(),
              name='Temperatura (MM7d)', line=dict(color='coral')),
    row=2, col=1
)

fig.update_yaxes(title_text='kWh', row=1, col=1)
fig.update_yaxes(title_text='°C', row=2, col=1)
fig.update_layout(height=500, title='Consumo e Temperatura ao Longo do Tempo', template='plotly_white')
fig.show()

## 7. Resumo Executivo

### Resultados Principais:
- Foram treinados 2 modelos: **Random Forest** e **LightGBM**
- Validação temporal com split treino (Jan-Mai) e teste (Jun) + expanding window CV
- As features mais importantes são os **lags de consumo** e **médias móveis**, indicando forte componente autoregressivo
- Variáveis climáticas (temperatura e umidade) contribuem marginalmente para o modelo
- Diferenças regionais são capturadas pelo modelo, com performance consistente entre regiões